# IG Content Extractor - Test Suite

**Project:** Instagram saved posts to markdown knowledge base

**Test all components step by step**

# 📖 Help / Usage Guide

**What this notebook does**

- Sets up all required system and Python packages.
- Clones the `Ig_content_extractor` repo from GitHub.
- Provides a step‑by‑step test of each component (OCR, Whisper, yt‑dlp, config, NIM AI).

**How to run the full extraction pipeline**

1. Run **Cell 2** and pick your `cookies.txt` (exported from your browser).
2. Run **Cell 3** — a paste box appears; paste your saved-post URLs as plain text (one per line), then press Enter. No file upload needed.
3. Run **Cell 4** and set your NVIDIA NIM API key.
4. Run the remaining test cells to verify the environment (Cell 11 makes one real NIM call).
5. Run **Cell 12 (Run the full pipeline)** — set the `COLLECTION` name first. Markdown files are written to the repo's `output/collections/<collection>/`.
6. Run **Cell 13** to download `output.zip` with all generated markdown back to your machine.

**Tips**

- If you see `Package 'libgl1-mesa-glx' has no installation candidate`, it's safe — `libgl1` is enough on newer Colab images.
- Cookies expire after a while; if downloads fail mid-run, re-export and re-upload `cookies.txt`.
- NIM free-tier rate limits are per-minute; the pipeline retries automatically.

In [ ]:
# 1. Upload cookies.txt (for private Instagram media)
# Pick your cookies.txt from your machine.
from google.colab import files
uploaded = files.upload()
with open('/content/cookies.txt', 'w') as f:
    for filename in uploaded:
        f.write(uploaded[filename])
print('cookies.txt uploaded to /content/cookies.txt')

In [ ]:
# 2. Paste your Instagram URLs here (plain text, one URL per line)

from google.colab import output
from IPython.display import display, HTML

_box = ('<div>Paste your URLs below — one per line. '
        'Only lines starting with http are used.</div>'
        "<textarea id='ig-urls' rows='10' style='width:100%;font-family:monospace;"
        "font-size:13px'></textarea>")
display(HTML(_box))

print('Paste your URLs into the box above, then press Enter here to continue.')
input()

raw = output.eval_js("document.getElementById('ig-urls').value") or ''
urls = []
for line in raw.replace('\r', '\n').split('\n'):
    for part in line.split():
        if part.startswith('http') and part not in urls:
            urls.append(part)

if not urls:
    raise ValueError('No URLs captured — paste them into the box, then press Enter.')

with open('/content/urls.txt', 'w') as f:
    f.write('\n'.join(urls) + '\n')
print(f'/content/urls.txt ready with {len(urls)} URLs')
print('\n'.join(urls[:10]) + ('\n...' if len(urls) > 10 else ''))


In [ ]:
# 3. Set your NVIDIA NIM API key
# Get one at https://build.nvidia.com (profile -> API Keys).

NVIDIA_NIM_API_KEY = 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  # <-- replace me

import os
os.environ['NVIDIA_NIM_API_KEY'] = NVIDIA_NIM_API_KEY
print('NIM key set:', NVIDIA_NIM_API_KEY[:8] + '...' if len(NVIDIA_NIM_API_KEY) > 10 else '(still the placeholder)')


In [9]:
# 4. Setup Colab Environment
!apt-get update -qq
!apt-get install -y -qq ffmpeg tesseract-ocr
%pip install -q yt-dlp openai-whisper pytesseract Pillow PyMuPDF pyyaml requests

print('✓ Environment ready!')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/noble/InRelease  503  Service Unavailable [IP: 185.125.189.186 443]
W: Some index files failed to download. They have been ignored, or old ones used instead.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 4.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.4 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 72.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.2 MB/s eta 0:00:00:00:0100:0

In [10]:
# 5. Clone project from GitHub
!git clone https://github.com/Deval2211/Ig_content_extractor.git /content/ig_content_extractor
import sys
sys.path.insert(0, '/content/ig_content_extractor')

print('✓ Project cloned')

Cloning into '/content/ig_content_extractor'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 31 (delta 8), reused 27 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 14.60 KiB | 1.83 MiB/s, done.
Resolving deltas: 100% (8/8), done.
✓ Project cloned


In [11]:
# 6. Setup directories
import os
os.makedirs('/content/tmp', exist_ok=True)
os.makedirs('/content/output/collections', exist_ok=True)

print('✓ Directories created')

✓ Directories created


In [12]:
# 7. Test OCR Pipeline (from Image_to_text)
from PIL import Image, ImageDraw
from extractors.ocr import extract_text_from_image, preprocess_image

# Create test image
img = Image.new('RGB', (600, 200), color='white')
draw = ImageDraw.Draw(img)
draw.text((50, 100), 'Test OCR from Image_to_text pipeline', fill='black')
img.save('/tmp/test_ocr.jpg')

result = extract_text_from_image('/tmp/test_ocr.jpg')
print(f'OCR Result: {result}')
print('✓ OCR pipeline working!')

OCR Result: Sette imnge, te a plowine
✓ OCR pipeline working!


In [13]:
# 8. Test Whisper
import whisper

print('Loading Whisper base model...')
model = whisper.load_model('base')
print('✓ Whisper model loaded!')

Loading Whisper base model...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 166MiB/s]


✓ Whisper model loaded!


In [15]:
# 9. Test yt-dlp
import yt_dlp
import pkg_resources
print(f'yt-dlp version: {pkg_resources.get_distribution("yt-dlp").version}')
print('✓ yt-dlp ready!')

yt-dlp version: 2026.8.19
✓ yt-dlp ready!


/tmp/ipykernel_4297/1149818682.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [16]:
# 10. Test pipeline components
import yaml
with open('/content/ig_content_extractor/config.yaml') as f:
    config = yaml.safe_load(f)

print('Config loaded:')
print(f'  AI Provider: {config["ai"]["provider"]}')
print(f'  Whisper model: {config["whisper"]["model"]}')
print('✓ Config working!')

Config loaded:
  AI Provider: openai
  Whisper model: base
✓ Config working!


In [ ]:
# 11. Test AI Writer with NVIDIA NIM (makes one real API call)

test_text = "This is a test transcript from an Instagram reel about machine learning and RAG systems."
test_url = "https://instagram.com/reel/test123"

result = generate_markdown(test_text, test_url, "reel", "test_collection", config)
print(result[:600])
print('...' if len(result) > 600 else '')
print('AI Writer working!')


In [ ]:
# 12. Run the full pipeline on your URLs
COLLECTION = 'my_collection'  # <-- name your output folder

# config.yaml is in the repo dir; cookies/urls/tmp/output live in /content
!cp /content/cookies.txt /content/ig_content_extractor/cookies.txt
!cd /content/ig_content_extractor && python pipeline.py \
    --collection "$COLLECTION" --urls /content/urls.txt

In [ ]:
# 13. Download your markdown knowledge base
!zip -r /content/output.zip /content/output/collections
from google.colab import files
files.download('/content/output.zip')